## 퀘스트 3(lasso) 버전이 더 좋음

In [1]:
# =====================================================
# Version 4: Advanced Lasso + ElasticNet + Ensemble
# =====================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Lasso, Ridge, ElasticNet
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, make_scorer


# =====================================================
# 1. 데이터 불러오기
# =====================================================

train_raw = pd.read_csv('modulabs-di/train.csv')
test_raw = pd.read_csv('modulabs-di/test.csv')

y = train_raw['SalePrice']
test_ids = test_raw['Id']

X = train_raw.drop(columns=['SalePrice', 'Id'])
X_test = test_raw.drop(columns=['Id'])


In [2]:
# =====================================================
# 2. Feature Engineering
# =====================================================

def make_features(df):
    df = df.copy()

    # 주택 전체 면적
    area_cols = [
        'TotalBsmtSF', '1stFlrSF', '2ndFlrSF'
    ]

    df['TotalSF'] = df[area_cols].sum(axis=1)

    # 전체 욕실 수
    df['TotalBath'] = (
        df['FullBath']
        + 0.5 * df['HalfBath']
        + df['BsmtFullBath']
        + 0.5 * df['BsmtHalfBath']
    )

    # 전체 현관 및 포치 면적
    porch_cols = [
        'OpenPorchSF', 'EnclosedPorch',
        '3SsnPorch', 'ScreenPorch'
    ]

    df['TotalPorchSF'] = df[porch_cols].sum(axis=1)

    # 주택 및 리모델링 연식
    df['HouseAge'] = df['YrSold'] - df['YearBuilt']
    df['RemodAge'] = df['YrSold'] - df['YearRemodAdd']

    # 품질과 면적의 결합
    df['QualityArea'] = (
        df['OverallQual'] * df['GrLivArea']
    )

    # 차고 면적과 주택 면적
    df['GarageAreaRatio'] = (
        df['GarageArea'] /
        (df['GrLivArea'] + 1)
    )

    # 지하실이 있는지
    df['HasBasement'] = (
        df['TotalBsmtSF'] > 0
    ).astype(int)

    # 차고가 있는지
    df['HasGarage'] = (
        df['GarageArea'] > 0
    ).astype(int)

    # 벽난로가 있는지
    df['HasFireplace'] = (
        df['Fireplaces'] > 0
    ).astype(int)

    return df


X = make_features(X)
X_test = make_features(X_test)


In [3]:
# =====================================================
# 3. 구조적 결측치 처리
# =====================================================

none_cols = [
    'Alley', 'BsmtQual', 'BsmtCond', 'BsmtExposure',
    'BsmtFinType1', 'BsmtFinType2', 'FireplaceQu',
    'GarageType', 'GarageFinish', 'GarageQual',
    'GarageCond', 'PoolQC', 'Fence', 'MiscFeature',
    'MasVnrType'
]

zero_cols = [
    'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF',
    'TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath',
    'GarageYrBlt', 'GarageCars', 'GarageArea',
    'MasVnrArea'
]

for col in none_cols:
    if col in X.columns:
        X[col] = X[col].fillna('None')
        X_test[col] = X_test[col].fillna('None')

for col in zero_cols:
    if col in X.columns:
        X[col] = X[col].fillna(0)
        X_test[col] = X_test[col].fillna(0)


In [4]:
# =====================================================
# 4. 전처리 파이프라인
# =====================================================

categorical_cols = X.select_dtypes(
    include=['object', 'category']
).columns.tolist()

numeric_cols = X.select_dtypes(
    include='number'
).columns.tolist()

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer([
    ('cat', categorical_pipeline, categorical_cols),
    ('num', numeric_pipeline, numeric_cols)
])



C:\Users\chaee\AppData\Local\Temp\ipykernel_32868\3095975559.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(


In [5]:
# =====================================================
# 5. 학습 / 검증 데이터 분리
# =====================================================

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

y_train_log = np.log1p(y_train)



In [6]:
# =====================================================
# 6. 로그 변환을 고려한 RMSE 평가 함수
# =====================================================

def original_rmse(y_true_log, y_pred_log):
    y_true = np.expm1(y_true_log)
    y_pred = np.expm1(y_pred_log)

    return -np.sqrt(
        mean_squared_error(y_true, y_pred)
    )


rmse_scorer = make_scorer(original_rmse)

In [8]:
# =====================================================
# 7. 모델 및 하이퍼파라미터 설정
# =====================================================

models = {

    'Lasso': (
        Lasso(max_iter=100000),
        {
            'regressor__alpha': [
                0.0001, 0.0003, 0.0005,
                0.001, 0.003, 0.005, 0.01
            ]
        }
    ),

    'Ridge': (
        Ridge(),
        {
            'regressor__alpha': [
                1, 5, 10, 20, 30, 50, 100
            ]
        }
    ),

    'ElasticNet': (
        ElasticNet(max_iter=100000),
        {
            'regressor__alpha': [
                0.0001, 0.0003, 0.0005,
                0.001, 0.003, 0.005
            ],
            'regressor__l1_ratio': [
                0.3, 0.5, 0.7, 0.9, 1.0
            ]
        }
    ),

    'GradientBoosting': (
        GradientBoostingRegressor(random_state=42),
        {
            'regressor__n_estimators': [
                100, 200, 300
            ],
            'regressor__learning_rate': [
                0.03, 0.05, 0.1
            ],
            'regressor__max_depth': [
                2, 3
            ]
        }
    )
}


In [9]:
# =====================================================
# 8. GridSearchCV로 모델별 최적 파라미터 탐색
# =====================================================

best_models = {}
results = {}
valid_predictions = {}

for name, (regressor, params) in models.items():

    print(f"\n===== {name} 학습 시작 =====")

    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', regressor)
    ])

    grid = GridSearchCV(
        pipeline,
        param_grid=params,
        scoring=rmse_scorer,
        cv=5,
        n_jobs=-1
    )

    grid.fit(X_train, y_train_log)

    best_model = grid.best_estimator_

    pred_log = best_model.predict(X_valid)
    pred = np.expm1(pred_log)
    pred = np.maximum(pred, 0)

    rmse = np.sqrt(
        mean_squared_error(y_valid, pred)
    )

    mae = mean_absolute_error(y_valid, pred)

    best_models[name] = best_model
    valid_predictions[name] = pred

    results[name] = {
        'RMSE': rmse,
        'MAE': mae,
        'Best Parameters': grid.best_params_
    }

    print("Best Parameters:", grid.best_params_)
    print(f"Validation RMSE: {rmse:.2f}")
    print(f"Validation MAE: {mae:.2f}")



===== Lasso 학습 시작 =====
Best Parameters: {'regressor__alpha': 0.0001}
Validation RMSE: 19624.21
Validation MAE: 13641.89

===== Ridge 학습 시작 =====
Best Parameters: {'regressor__alpha': 1}
Validation RMSE: 23499.84
Validation MAE: 15365.95

===== ElasticNet 학습 시작 =====
Best Parameters: {'regressor__alpha': 0.0005, 'regressor__l1_ratio': 0.3}
Validation RMSE: 22610.74
Validation MAE: 14424.96

===== GradientBoosting 학습 시작 =====
Best Parameters: {'regressor__learning_rate': 0.1, 'regressor__max_depth': 2, 'regressor__n_estimators': 300}
Validation RMSE: 21673.35
Validation MAE: 15435.88


In [10]:
# =====================================================
# 9. 모델별 성능 비교
# =====================================================

results_df = pd.DataFrame(results).T

results_df = results_df.sort_values('RMSE')

print("\n===== 모델별 성능 비교 =====")
print(results_df.to_string())


===== 모델별 성능 비교 =====
                          RMSE           MAE                                                                               Best Parameters
Lasso             19624.212027   13641.88879                                                                  {'regressor__alpha': 0.0001}
GradientBoosting  21673.354428  15435.882463  {'regressor__learning_rate': 0.1, 'regressor__max_depth': 2, 'regressor__n_estimators': 300}
ElasticNet         22610.73561  14424.956395                                      {'regressor__alpha': 0.0005, 'regressor__l1_ratio': 0.3}
Ridge             23499.835159  15365.945909                                                                       {'regressor__alpha': 1}


In [11]:
# =====================================================
# 10. Weighted Blending
# =====================================================

# Lasso, Ridge, ElasticNet 예측 결합

blend_pred = (
    0.5 * valid_predictions['Lasso']
    + 0.2 * valid_predictions['Ridge']
    + 0.3 * valid_predictions['ElasticNet']
)

blend_rmse = np.sqrt(
    mean_squared_error(y_valid, blend_pred)
)

blend_mae = mean_absolute_error(y_valid, blend_pred)

print("\n===== Weighted Blending =====")
print(f"Blending RMSE: {blend_rmse:.2f}")
print(f"Blending MAE: {blend_mae:.2f}")



===== Weighted Blending =====
Blending RMSE: 20994.19
Blending MAE: 14094.73


In [12]:
# =====================================================
# 11. 최종 모델 선택
# =====================================================

all_scores = {
    name: values['RMSE']
    for name, values in results.items()
}

all_scores['Weighted Blending'] = blend_rmse

best_name = min(all_scores, key=all_scores.get)

print("\n===== 최종 모델 =====")
print("Best Model:", best_name)
print("Best Validation RMSE:", round(all_scores[best_name], 2))


===== 최종 모델 =====
Best Model: Lasso
Best Validation RMSE: 19624.21


In [13]:
# =====================================================
# 12. 전체 학습 데이터로 재학습
# =====================================================

if best_name == 'Weighted Blending':

    final_models = best_models

    for model in final_models.values():
        model.fit(X, np.log1p(y))

else:
    final_model = best_models[best_name]
    final_model.fit(X, np.log1p(y))


# =====================================================
# 13. 테스트 데이터 예측
# =====================================================

if best_name == 'Weighted Blending':

    test_predictions = {}

    for name, model in final_models.items():
        pred_log = model.predict(X_test)
        test_predictions[name] = np.expm1(pred_log)

    test_pred = (
        0.5 * test_predictions['Lasso']
        + 0.2 * test_predictions['Ridge']
        + 0.3 * test_predictions['ElasticNet']
    )

else:

    test_pred_log = final_model.predict(X_test)
    test_pred = np.expm1(test_pred_log)


test_pred = np.maximum(test_pred, 0)


In [14]:
# =====================================================
# 14. 제출 파일 생성
# =====================================================

submission_v4 = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': test_pred
})

submission_v4.to_csv(
    'submission_v4_advanced.csv',
    index=False
)

print("\n제출 파일 생성 완료!")
print(submission_v4.head())
print("파일 크기:", submission_v4.shape)


제출 파일 생성 완료!
    Id      SalePrice
0   86  257198.153172
1  227  266369.646588
2  492  140076.325219
3   64  118049.855825
4  576  117544.595769
파일 크기: (292, 2)
